# 07. 품질 평가 — precision@k / MRR

**무엇을 하나:** 정답을 아는 쿼리 묶음(`scripts/rag_eval.py` 의 16쿼리 = 오타4 + 묘사8 + 구어4)으로 검색 품질을 수치화한다.
- **P@1 / P@3**: 정답이 1위 / 상위 3 안에 든 쿼리 비율.
- **MRR**: 정답 순위의 역수 평균(1위=1.0, 2위=0.5, 3위=0.33...). 높을수록 좋다.

**PLAIN vs SMART:** PLAIN = 순수 벡터, SMART = HyDE/Fusion(LLM). **SMART 는 LLM 이라 run 마다 수치가 변동**한다(결정적이지 않음) — 한 번의 수치로 우열을 단정하지 말 것.

**정본/주의:** 하네스 `scripts/rag_eval.py`, 해석은 `docs/rag-experiment-report.md`(§1-1 SMART 비결정성), 문서 종합은 `docs/rag-INDEX.md`.

In [ ]:
import sys,os,asyncio
from pathlib import Path
B=(Path.cwd().parent/'backend') if Path.cwd().name=='notebooks' else Path.cwd()/'backend'
B=B.resolve(); sys.path.insert(0,str(B)); os.chdir(B); sys.path.insert(0,str(B/'scripts'))
try: sys.stdout.reconfigure(encoding='utf-8')
except Exception: pass
from dotenv import load_dotenv; load_dotenv()
from rag_eval import EVAL, _rank_of            # 정본 16쿼리 + 랭크 계산 재사용
from app.rag.retriever import search_recipes
from app.rag.smart_search import smart_search

async def ev(fn):
    p1=p3=0; rr=0.0
    for q,exp,cat in EVAL:
        docs=await fn(q,k=3); r=_rank_of(docs,exp)
        p1+=r==1; p3+= r in (1,2,3); rr+=(1.0/r if r else 0.0)
    n=len(EVAL); return p1/n,p3/n,rr/n

pl=asyncio.run(ev(search_recipes)); sm=asyncio.run(ev(smart_search))
print('PLAIN  P@1=%.0f%% P@3=%.0f%% MRR=%.3f'%(pl[0]*100,pl[1]*100,pl[2]))
print('SMART  P@1=%.0f%% P@3=%.0f%% MRR=%.3f  (HyDE/Fusion LLM -> run마다 변동)'%(sm[0]*100,sm[1]*100,sm[2]))
print('정본 하네스: scripts/rag_eval.py | 보고서: docs/rag-experiment-report.md (§1-1 SMART 비결정성)')